# Yonerwin Rodriguez - Actividad de Semana 1

Yonerwin Rodriguez Semana 1

In [2]:
# Paso 1:  archivo LAS para convertirlo a un dataframe
# Aquí cargo el z LAS que contiene los datos del pozo
import lasio
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


# Especifico la ruta del archivo LAS que quiero analizar.
las_file_path = "data/well_logs/WA1.LAS"
las = lasio.read(las_file_path)


# Convierto los datos del archivo LAS a un DataFrame de pandas 
df = las.df().dropna()

# Mostrar las primeras filas del DataFrame
# Verifico que los datos se hayan cargado correctamente.
df.head()

,SP,GR,CALI,BITSIZE,LL8,ILM,ILD,RHOB,NPHI,DT,MUDWGT
M__DEPTH,,,,,,,,,,,
112.5,-1.07405,67.0721,13.4333,12.25,7.2406,3.4136,3.3481,2.2708,40.3011,144.6073,9.0
113.0,0.50042,67.4492,13.2112,12.25,5.8327,3.3976,2.9712,2.2603,39.3173,144.2272,9.0
113.5,-0.71472,66.5118,12.7873,12.25,5.1744,3.3719,2.9776,2.3074,38.2452,143.7870,9.0
114.0,-2.41777,63.5625,12.8964,12.25,6.0517,3.3336,3.3151,2.3573,37.3958,143.5484,9.0
114.5,-3.97690,59.6142,12.9686,12.25,8.2100,3.2956,3.4569,2.3555,37.5981,143.4810,9.0


In [3]:
# Paso 2
# Aquí calculo estadísticas descriptivas para entender mejor los datos del pozo
# Esto me permite identificar valores como el promedio, mínimo, máximo y otros
statistics = df.describe()


# Reviso las estadísticas para identificar tendencias o valores interesantes
statistics

,SP,GR,CALI,BITSIZE,LL8,ILM,ILD,RHOB,NPHI,DT,MUDWGT
count,3349.000000,3349.000000,3349.000000,3349.000000,3349.000000,3349.000000,3349.000000,3349.000000,3349.000000,3349.000000,3349.000000
mean,-12.759796,86.630680,13.003942,12.235443,17.933521,10.879907,14.378462,2.446782,35.403041,116.768313,9.684951
std,9.212121,13.860863,0.633663,0.233220,10.477056,5.496044,8.865627,0.057432,3.237050,13.358044,0.239437
min,-41.235570,53.101800,8.709800,8.500000,3.437500,2.620900,2.971200,2.036700,23.887300,69.402400,9.000000
25%,-17.489730,80.224000,12.731100,12.250000,12.944000,8.804000,11.269500,2.417900,33.258100,105.628500,9.600000
50%,-10.285140,86.497000,12.964800,12.250000,15.739100,10.403000,13.542000,2.454200,35.291100,116.729600,9.800000
75%,-5.899580,91.357600,13.253100,12.250000,20.030800,13.048800,16.800900,2.484800,37.190300,127.156100,9.800000
max,8.850300,226.662200,15.961800,12.250000,111.520400,65.611300,126.788200,2.584200,58.582200,163.841400,9.900000


In [4]:
# Paso 3:

# Aquí calculo la concentración volumétrica de lutita usando el registro Gamma Ray

# Esto es importante para identificar las zonas con mayor contenido de lutita

# Definir los valores mínimos y máximos del registro gamma ray
# Calculo los valores mínimo y máximo del registro GR para usarlos en la fórmula.
GR_min = df['GR'].min()

GR_max = df['GR'].max()

# Aplico la fórmula para calcular V_sh y lo añado como una nueva columna en el DataFrame.
df['V_sh'] = (df['GR'] - GR_min) / (GR_max - GR_min)


# Verifico que la columna V_sh se haya calculado correctamente.
df[['GR', 'V_sh']].head()

,GR,V_sh
M__DEPTH,,
112.5,67.0721,0.080492
113.0,67.4492,0.082665
113.5,66.5118,0.077264
114.0,63.5625,0.060271
114.5,59.6142,0.037522


In [5]:
# Paso 4
# Aquí identifico las propiedades de la lutita pura y calculo las porosidades corregidas.
# Encuentro los valores máximos de GR y sus correspondientes valores de RHOB y NPHI.
GR_sh = df['GR'].max()
RHOB_sh = df.loc[df['GR'] == GR_sh, 'RHOB'].values[0]
NPHI_sh = df.loc[df['GR'] == GR_sh, 'NPHI'].values[0]

# Uso las fórmulas dadas para calcular las porosidades corregidas.
df['phi_D'] = (df['RHOB'] - 1.65) / (2.65 - 1.65)
df['phi_D_c'] = (df['phi_D'] - df['V_sh'] * (RHOB_sh - 1.65) / (2.65 - 1.65)) / (1 - df['V_sh'])
df['phi_N_c'] = (df['NPHI'] - df['V_sh'] * NPHI_sh) / (1 - df['V_sh'])
df['phi'] = ((df['phi_D_c']**2 + df['phi_N_c']**2) / 2)**0.5

# Verifico que las porosidades corregidas se hayan calculado correctamente.
df[['GR', 'RHOB', 'NPHI', 'V_sh', 'phi_D', 'phi_D_c', 'phi_N_c', 'phi']].head()

,GR,RHOB,NPHI,V_sh,phi_D,phi_D_c,phi_N_c,phi
M__DEPTH,,,,,,,,
112.5,67.0721,2.2708,40.3011,0.080492,0.6208,0.616152,39.741991,28.105208
113.0,67.4492,2.2603,39.3173,0.082665,0.6103,0.604569,38.653084,27.335201
113.5,66.5118,2.3074,38.2452,0.077264,0.6574,0.656018,37.538244,26.547600
114.0,63.5625,2.3573,37.3958,0.060271,0.7073,0.709442,36.799821,26.026238
114.5,59.6142,2.3555,37.5981,0.037522,0.7055,0.706732,37.243725,26.340031


In [ ]:
# Paso  5
# aquí creo los track para ver los datos del pozo
fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(20, 10), sharey=True)

# Track 1: Gamma Ray y V_sh
# Visualizo el registro Gamma Ray junto con la concentración volumétrica de lutita.
axes[0].plot(df['GR'], df.index, label='Gamma Ray (GR)', color='green')
axes[0].plot(df['V_sh'], df.index, label='V_sh', color='blue')
axes[0].set_xlabel('GR / V_sh')
axes[0].set_ylabel('Depth')
axes[0].invert_yaxis()
axes[0].legend()
axes[0].grid()

# Track 2: RHOB y NPHI
# Comparo la densidad y la porosidad neutrónica para identificar tendencias.
axes[1].plot(df['RHOB'], df.index, label='RHOB', color='red')
axes[1].plot(df['NPHI'], df.index, label='NPHI', color='purple')
axes[1].set_xlabel('RHOB / NPHI')
axes[1].invert_yaxis()
axes[1].legend()
axes[1].grid()

# Track 3: Resistividad (LL8, ILM, ILD)
# Uso una escala logarítmica para visualizar los registros de resistividad.
axes[2].semilogx(df['LL8'], df.index, label='LL8 (Shallow)', color='orange')
axes[2].semilogx(df['ILM'], df.index, label='ILM (Medium)', color='brown')
axes[2].semilogx(df['ILD'], df.index, label='ILD (Deep)', color='black')
axes[2].set_xlabel('Resistivity (log scale)')
axes[2].invert_yaxis()
axes[2].legend()
axes[2].grid()

# Track 4: Otros registros interesantes
# Aquí incluyo un registro adicional para análisis complementario.
axes[3].plot(df['CALI'], df.index, label='CALI (Caliper)', color='cyan')
axes[3].set_xlabel('CALI')
axes[3].invert_yaxis()
axes[3].legend()
axes[3].grid()

# Uso tight_layout para asegurarme de que los gráficos no se solapen.
plt.tight_layout()
plt.show()

In [ ]:
# Paso 6
# Determinar resistividad del agua (R_w) y resistividad de la arena (R_s)
R_w = df['ILD'].min()  # Resistividad en formación de agua pura
R_s = df['ILD'].max()  # Resistividad en formación de arena pura

# Calcular propiedades petrofísicas
df['phi_T'] = (1 - df['V_sh']) * df['phi'] + df['V_sh'] * NPHI_sh
df['S_w'] = (R_w / (R_s * df['phi']**2))**0.5
A, B, C = 1, 2, 2  # Constantes para el cálculo de permeabilidad
df['k'] = (A * df['phi']**B) / (df['S_w']**C)
df['HPV'] = (1 - df['S_w']) * df['phi']

# Mostrar las primeras filas con los nuevos cálculos
df[['phi_T', 'S_w', 'k', 'HPV']].head()

In [ ]:
# Paso 7: Crear crossplots (gráficos de dispersión 3x3)
import seaborn as sns

properties = ['GR', 'RHOB', 'NPHI', 'V_sh', 'phi', 'k', 'S_w']

# Crear una figura para los crossplots
sns.pairplot(df[properties], diag_kind='kde', plot_kws={'alpha': 0.5})
plt.suptitle('Crossplots de propiedades de registros', y=1.02)
plt.show()

In [ ]:
# Paso 8: Gráfico final del conjunto de registros con verificación de columnas
fig, axes = plt.subplots(nrows=1, ncols=8, figsize=(30, 10), sharey=True)

# Track 1: Gamma Ray y V_sh
if 'GR' in df.columns:
    axes[0].plot(df['GR'], df.index, label='Gamma Ray (GR)', color='green')
if 'V_sh' in df.columns:
    axes[0].plot(df['V_sh'], df.index, label='V_sh', color='blue')
axes[0].set_xlabel('GR / V_sh')
axes[0].set_ylabel('Depth')
axes[0].invert_yaxis()
axes[0].legend()
axes[0].grid()

# Track 2: RHOB y NPHI
if 'RHOB' in df.columns:
    axes[1].plot(df['RHOB'], df.index, label='RHOB', color='red')
if 'NPHI' in df.columns:
    axes[1].plot(df['NPHI'], df.index, label='NPHI', color='purple')
axes[1].set_xlabel('RHOB / NPHI')
axes[1].invert_yaxis()
axes[1].legend()
axes[1].grid()

# Track 3: Resistividad (LL8, ILM, ILD)
if 'LL8' in df.columns:
    axes[2].semilogx(df['LL8'], df.index, label='LL8 (Shallow)', color='orange')
if 'ILM' in df.columns:
    axes[2].semilogx(df['ILM'], df.index, label='ILM (Medium)', color='brown')
if 'ILD' in df.columns:
    axes[2].semilogx(df['ILD'], df.index, label='ILD (Deep)', color='black')
axes[2].set_xlabel('Resistivity (log scale)')
axes[2].invert_yaxis()
axes[2].legend()
axes[2].grid()

# Track 4: Otros registros interesantes
if 'CALI' in df.columns:
    axes[3].plot(df['CALI'], df.index, label='CALI (Caliper)', color='cyan')
axes[3].set_xlabel('CALI')
axes[3].invert_yaxis()
axes[3].legend()
axes[3].grid()

# Track 5: Porosidad total
if 'phi_T' in df.columns:
    axes[4].plot(df['phi_T'], df.index, label='Porosidad Total (phi_T)', color='magenta')
axes[4].set_xlabel('Porosidad Total')
axes[4].invert_yaxis()
axes[4].legend()
axes[4].grid()

# Track 6: Permeabilidad
if 'k' in df.columns:
    axes[5].semilogx(df['k'], df.index, label='Permeabilidad (k)', color='brown')
axes[5].set_xlabel('Permeabilidad (log scale)')
axes[5].invert_yaxis()
axes[5].legend()
axes[5].grid()

# Track 7: Saturación
if 'S_w' in df.columns:
    axes[6].fill_betweenx(df.index, 0, df['S_w'], color='blue', alpha=0.5, label='Saturación de Agua (S_w)')
    axes[6].fill_betweenx(df.index, df['S_w'], 1, color='red', alpha=0.5, label='Saturación de Hidrocarburos')
axes[6].set_xlabel('Saturación')
axes[6].invert_yaxis()
axes[6].legend()
axes[6].grid()

# Track 8: Volumen poroso de hidrocarburos
if 'HPV' in df.columns:
    axes[7].plot(df['HPV'], df.index, label='HPV', color='green')
axes[7].set_xlabel('HPV')
axes[7].invert_yaxis()
axes[7].legend()
axes[7].grid()

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

3
